# Diffusion-based Weather Downscaling Visualization Notebook

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import string

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib as mpl
import properscoring as ps
import matplotlib.pyplot as plt

from pathlib import Path

from tqdm.auto import tqdm
from utils.mask import land
from datetime import timedelta
from matplotlib.patches import Patch
from scipy.interpolate import griddata
from utils.presets import Precipitation
from utils.download import west_us_extent as my_extent
from utils.hrrr import download_hrrr

cmap, norm = Precipitation.cmap_norm()

labels = {
    'gc': 'GraphCast',
    'anen': 'AnEn',
    'anen_ens': 'AnEn',
    'unet': 'Unet',
    'cd': 'CorrDiff',
    'cd_ens': 'CorrDiff',
    'obs': 'PRISM',
}

colors = {
    'gc': '#fe5f55',
    'anen': '#E09F3E',
    'anen_ens': '#E09F3E',
    'unet': '#94C9A9',
    'cd': '#777DA7',
    'cd_ens': '#777DA7',
    'obs': 'black',
}

figure_dir = Path('./Figures_ED')

with xr.open_zarr('/home/myid/wh63884/data/derived/PRISM/PRISM_daily_stable_westus_20190101_20241231.zarr') as ds:
    grids_x = ds.x.to_numpy()
    grids_y = ds.y.to_numpy()

## Load Data

In [3]:
%%time

output_dir = '/home/myid/wh63884/github/2025_Diffusion/lightning_logs/CorrDiff/'
anen_path = '/home/myid/wh63884/github/2025_Diffusion/tmp/AnEn_t2_pr_test2024.nc'
start_with = 'prediction'

# List to store full paths
matched_files = []

# Walk through the directory tree
for dirpath, dirnames, filenames in os.walk(output_dir):
    for filename in filenames:
        if filename.startswith(start_with):
            full_path = os.path.join(dirpath, filename)
            matched_files.append(full_path)

matched_files = sorted(matched_files)
assert len(matched_files) > 0, "No matching files have been found!"

results = {
    'lead_time': [],
    'obs': [],
    'gc': [],
    'unet': [],
    'cd': [],
    'cd_ens': [],
}

for f in matched_files:
    d = np.load(f)
    for k in results.keys():
        results[k].append(d[k])

for k in results.keys():
    if len(results[k][0].shape) == 0:
        results[k] = np.stack(results[k])
    elif len(results[k][0].shape) == 1:
        results[k] = np.concatenate(results[k], axis=0)
    elif len(results[k][0].shape) == 4:
        results[k] = np.concatenate(results[k], axis=1)
    elif len(results[k][0].shape) == 5:
        results[k] = np.concatenate(results[k], axis=2)
    else:
        raise Exception(f'Unexpected shape for {k}: {results[k][0].shape}')

results['forecast_init_time'] = pd.to_datetime(d['forecast_init_times']).date
results['valid_time'] = pd.to_datetime(d['valid_times']).date

with xr.open_dataset(anen_path) as ds:

    anen = (ds['analogs']
        .sel(num_times=results['forecast_init_time'])
        .transpose('num_analogs', 'num_times', 'num_flts', 'num_stations')
        .to_numpy()
    )
    
    anen = anen.reshape((
        anen.shape[0], anen.shape[1], anen.shape[2],
        results['unet'].shape[2], results['unet'].shape[3]
    ))

    # Select the first 15 members to be consistent with CD
    anen = anen[:15]

    results['anen'] = np.mean(anen, axis=0)
    results['anen_ens'] = anen

for k, e in results.items():
    print(f'{k}: {e.shape}')

lead_time: (7,)
obs: (366, 7, 228, 216)
gc: (366, 7, 228, 216)
unet: (366, 7, 228, 216)
cd: (366, 7, 228, 216)
cd_ens: (15, 366, 7, 228, 216)
forecast_init_time: (366,)
valid_time: (366,)
anen: (366, 7, 228, 216)
anen_ens: (15, 366, 7, 228, 216)
CPU times: user 12.1 s, sys: 13.5 s, total: 25.6 s
Wall time: 26.5 s


## Downscaled Examples

In [4]:
hrrr_mask = land(grids_x, grids_y, 'tmp/hrrr_mask.pkl')

In [155]:
%%time

selected_times = [
    pd.to_datetime('2024-02-15').date(),
    pd.to_datetime('2024-02-01').date(),
    pd.to_datetime('2024-02-02').date(),
    pd.to_datetime('2024-02-03').date(),
    pd.to_datetime('2024-02-04').date(),
    pd.to_datetime('2024-01-01').date(),
    pd.to_datetime('2024-01-10').date(),
    pd.to_datetime('2024-01-02').date(),
]

for valid_time in tqdm(selected_times, leave=False):

    out_file = figure_dir / f'DownscaledMap_{valid_time}_revised.png'
    
    verify_keys = ['gc', 'anen', 'unet', 'cd']
    
    obs_lead_idx = 0
    obs_init_idx = np.where(results['forecast_init_time'] + pd.to_timedelta(obs_lead_idx + 1, unit='d') == valid_time)[0].item()
    obs_img = results['obs'][obs_init_idx, obs_lead_idx]
    
    fig = plt.figure(figsize=(12, 5.9))
    gs = mpl.gridspec.GridSpec(4, 8)
    
    axes = []
    
    corner_text_margins = (0.03, 0.03)

    # PRISM
    ax = fig.add_subplot(gs[:2, 0])
    ax.imshow(obs_img, cmap=cmap, norm=norm)
    text = f"(0)\n{labels['obs']}\nMax: {np.nanmax(results['obs'][obs_init_idx, obs_lead_idx]):0.3f}\nValid: {valid_time}"
    ax.text(*corner_text_margins, text, transform=ax.transAxes)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

    # HRRR
    hrrr = download_hrrr(valid_time, lead_time=1, grids_x=grids_x, grids_y=grids_y)
    ax = fig.add_subplot(gs[2:3, 0])
    hrrr[~hrrr_mask] = np.nan
    ax.imshow(hrrr, cmap=cmap, norm=norm)
    err = (np.nanmean((hrrr - obs_img) ** 2)) ** 0.5
    text = f"(29)\nHRRR (LD 1)\nMax: {np.nanmax(hrrr):0.3f}\nRMSE: {err:0.3f}"
    ax.text(*corner_text_margins, text, transform=ax.transAxes)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    
    hrrr = download_hrrr(valid_time, , lead_time=2, grids_x=grids_x, grids_y=grids_y)
    ax = fig.add_subplot(gs[3:, 0])
    hrrr[~hrrr_mask] = np.nan
    ax.imshow(hrrr, cmap=cmap, norm=norm)
    err = (np.nanmean((hrrr - obs_img) ** 2)) ** 0.5
    text = f"(30)\nHRRR (LD 2)\nMax: {np.nanmax(hrrr):0.3f}\nRMSE: {err:0.3f}"
    ax.text(*corner_text_margins, text, transform=ax.transAxes)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    
    for row_idx, k in enumerate(verify_keys):
        for col_idx in range(7):
            ax = fig.add_subplot(gs[row_idx, col_idx + 1])
    
            if row_idx == 0:
                ax.set_title(f'Lead day {col_idx + 1}')
    
            lead_idx = col_idx
            init_idx = np.where(results['forecast_init_time'] + pd.to_timedelta(lead_idx + 1, unit='d') == valid_time)[0].item()
            img = results[k][init_idx, lead_idx]
            
            cbar = ax.imshow(img, cmap=cmap, norm=norm)
    
            text = f"({row_idx * 7 + col_idx + 1})\nMax: {np.nanmax(results[k][init_idx, lead_idx]):0.3f}\n"
    
            err = np.sqrt(np.nanmean((img - obs_img) ** 2))
            text += f'RMSE: {err:0.3f}'
    
            ax.text(*corner_text_margins, text, transform=ax.transAxes)
            ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    
            axes.append(ax)
            
        ax.yaxis.set_label_position('right')
        ax.set_ylabel(labels[k], va='center', rotation='vertical', labelpad=10)
    
    fig.subplots_adjust(left=0.01, right=0.91, bottom=0.02, top=0.95, wspace=0, hspace=0)
    cbar_ax = fig.add_axes([0.939, 0.02, 0.01, 0.93])
    cbar = fig.colorbar(cbar, cax=cbar_ax)
    cbar.set_label('Precipitation [$mm$]')
        
    plt.savefig(out_file, dpi=300)
    plt.close(fig)

## RMSE, CRPS, Bias by Lead Times

In [4]:
from tqdm.contrib.concurrent import process_map
from functools import partial

In [5]:
download_hrrr_ld1 = partial(download_hrrr, lead_time=1, grids_x=grids_x, grids_y=grids_y)
download_hrrr_ld2 = partial(download_hrrr, lead_time=2, grids_x=grids_x, grids_y=grids_y)

ld1 = process_map(download_hrrr_ld1, [i + timedelta(days=1) for i in results['forecast_init_time']], chunksize=1, max_workers=10)
ld2 = process_map(download_hrrr_ld2, [i + timedelta(days=2) for i in results['forecast_init_time']], chunksize=1, max_workers=10)

hrrr = np.stack([ld1, ld2], axis=1)

  0%|          | 0/366 [00:00<?, ?it/s]

/home/myid/wh63884/miniconda3/envs/venv_torch/lib/python3.11/site-packages/herbie/core.py:1280: UserWarning: Will not remove GRIB file because it previously existed.
  warnings.warn("Will not remove GRIB file because it previously existed.")
/home/myid/wh63884/miniconda3/envs/venv_torch/lib/python3.11/site-packages/herbie/core.py:1280: UserWarning: Will not remove GRIB file because it previously existed.
  warnings.warn("Will not remove GRIB file because it previously existed.")
/home/myid/wh63884/miniconda3/envs/venv_torch/lib/python3.11/site-packages/herbie/core.py:1280: UserWarning: Will not remove GRIB file because it previously existed.
  warnings.warn("Will not remove GRIB file because it previously existed.")


  0%|          | 0/366 [00:00<?, ?it/s]

/home/myid/wh63884/miniconda3/envs/venv_torch/lib/python3.11/site-packages/herbie/core.py:1280: UserWarning: Will not remove GRIB file because it previously existed.
  warnings.warn("Will not remove GRIB file because it previously existed.")
/home/myid/wh63884/miniconda3/envs/venv_torch/lib/python3.11/site-packages/herbie/core.py:1280: UserWarning: Will not remove GRIB file because it previously existed.
  warnings.warn("Will not remove GRIB file because it previously existed.")
/home/myid/wh63884/miniconda3/envs/venv_torch/lib/python3.11/site-packages/herbie/core.py:1280: UserWarning: Will not remove GRIB file because it previously existed.
  warnings.warn("Will not remove GRIB file because it previously existed.")


In [6]:
err_table = []
obs = results['obs'][:, :2]
for arr in [results['cd'][:, :2], results['anen'][:, :2], hrrr]:
    err = (np.nanmean((arr - obs) ** 2, axis=(0, 2, 3))) ** 0.5
    err_table.append(err)

In [7]:
pd.DataFrame(err_table, columns=['Lead Day 1', 'Lead Day 2'], index=['CorrDiff', 'AnEn', 'HRRR']).transpose()

,CorrDiff,AnEn,HRRR
Lead Day 1,2.392164,2.346173,3.154857
Lead Day 2,2.675743,2.635255,3.748696


In [181]:
%%time

# CRPS
crps = {}

crps['gc'] = np.nanmean(
    np.abs(results['gc'] - results['obs']),
    axis=(2, 3)
)

crps['unet'] = np.nanmean(
    np.abs(results['unet'] - results['obs']),
    axis=(2, 3)
)

verify_keys = ['anen_ens', 'cd_ens']

for k in verify_keys:
    crps[k] = np.nanmean(
        ps.crps_ensemble(results['obs'], results[k], axis=0),
        axis=(2, 3)
    )

crps = {k: crps[k] for k in ['gc', 'anen_ens', 'unet', 'cd_ens']}

# RMSE
verify_keys = ['gc', 'anen', 'unet', 'cd']
rmse = {
    k: np.sqrt(np.nanmean(
        (results[k] - results['obs']) ** 2,
        axis=(2, 3)
    )) for k in verify_keys
}

# Bias
verify_keys = ['gc', 'anen', 'unet', 'cd']
bias = {
    k: np.nanmean(
        results[k] - results['obs'],
        axis=(2, 3)
    ) for k in verify_keys
}

CPU times: user 34.8 s, sys: 4min 1s, total: 4min 36s
Wall time: 4min 36s


In [44]:
def shade(hex_color, x):
    """
    Create a shade of a color.
    x = 1.0 → original color
    x = 0.0 → black
    x > 1.0 → brighter (tint toward white if you clamp)
    """
    hex_color = hex_color.lstrip('#')
    r, g, b = tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    
    r = int(max(0, min(255, r * x)))
    g = int(max(0, min(255, g * x)))
    b = int(max(0, min(255, b * x)))
    
    return f'#{r:02x}{g:02x}{b:02x}'

In [156]:
n_leads = 7
positions = np.arange(n_leads)
width = 0.17

def bplot(d, ax, ylabel, keys=None, width=width):

    if keys is None:
        keys = d.keys()

    legend_handles = []
    legend_labels = []
    
    for i, key in enumerate(keys):

        if key in labels:
            label_key = labels[key]
            color_key = colors[key]
        elif key.startswith('exp'):
            label_key = labels['anen'] + ' ' + key.replace('exp', 'Exp. ')
            color_key = shade(colors['anen'], 0.2+0.1 * float(label_key[-1]))
        else:
            raise Exception(f'Unknown key: {key}')
        
        data = [d[key][:, l] for l in range(n_leads)]
        print(f'{label_key}: {[np.median(d[key][:, l]).item() for l in range(n_leads)]}')
        
        # Shift positions for this method
        pos = positions + (i - (len(verify_keys)-1)/2) * width
        bplot = ax.boxplot(
            data,
            positions=pos,
            widths=width*0.9,
            showfliers=False,
            patch_artist=True,
            medianprops=dict(color='black'),
            zorder=102,
        )
    
        for patch in bplot['boxes']:
            patch.set_facecolor(color_key)
            
        legend_handles.append(Patch(facecolor=color_key))
        legend_labels.append(label_key)
    
    ax.set_xticks(positions, [f"{l+1}" for l in range(n_leads)])
    ax.set_ylabel(ylabel)
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    lgd = ax.legend(legend_handles, legend_labels, loc="upper left", ncols=2)
    lgd.set_zorder(103)
    ax.set_xlim(-width * 3, positions[-1] + width * 3)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(7, 8))

print('RMSE medians')
bplot(rmse, axes[0], 'RMSE [$mm$]')

print('CRPS medians')
bplot(crps, axes[1], 'CRPS [$mm$]')

print('Bias medians')
bplot(bias, axes[2], 'Bias [$mm$]')

axes[2].set_xlabel("Lead Day")
axes[2].plot([-10, 20], [0, 0], c='darkred', zorder=101, linestyle='dashed')

axes[0].text(0.99, 0.99, '(a)', ha='right', va='top', transform=axes[0].transAxes)
axes[1].text(0.99, 0.99, '(b)', ha='right', va='top', transform=axes[1].transAxes)
axes[2].text(0.99, 0.99, '(c)', ha='right', va='top', transform=axes[2].transAxes)

fig.subplots_adjust(left=0.10, right=0.99, bottom=0.06, top=0.99, hspace=0.03)
plt.savefig(figure_dir / 'crps_rmse_bias_by_leads_boxplot.png', dpi=300)
plt.close(fig)

RMSE medians
GraphCast: [0.4313417077064514, 0.43885451555252075, 0.4445475935935974, 0.49136239290237427, 0.5369687080383301, 0.6153775453567505, 0.7404618263244629]
AnEn: [0.42952499761892615, 0.46774774126248253, 0.4817936359231181, 0.5488760670780228, 0.6237917952841321, 0.7734931654573507, 0.8978342917844944]
Unet: [0.406706839799881, 0.4372735321521759, 0.4320976734161377, 0.5027782917022705, 0.616836428642273, 0.7385650277137756, 0.7340937852859497]
CorrDiff: [0.42159366607666016, 0.468960702419281, 0.4333403706550598, 0.5716004371643066, 0.5464454889297485, 0.5826785564422607, 0.6313464641571045]
CRPS medians
GraphCast: [0.15314069390296936, 0.13937954604625702, 0.14991524815559387, 0.1759311854839325, 0.19838660955429077, 0.20427921414375305, 0.25906792283058167]
AnEn: [0.07169999029989518, 0.075876967394088, 0.07616394518641387, 0.08826243136693794, 0.09265333800015879, 0.11563931713126226, 0.12730266930701345]
Unet: [0.12747645378112793, 0.13674159348011017, 0.13008987903594

In [7]:
# fig, axes = plt.subplots(1, 3, figsize=(10, 3))

# for k, v in rmse.items():
#     axes[0].plot(range(1, n_leads+1), v.mean(axis=0), label=labels[k], c=colors[k], marker='.')
#     axes[0].set_ylabel('RMSE [$mm$]')
#     axes[0].set_ylim(1.0, 3)

# for k, v in crps.items():
#     axes[1].plot(range(1, n_leads+1), v.mean(axis=0), label=labels[k], c=colors[k], marker='.')
#     axes[1].set_ylabel('CRPS [$mm$]')
#     axes[1].set_ylim(0.25, 1.75)

# for k, v in bias.items():
#     axes[2].plot(range(1, n_leads+1), v.mean(axis=0), label=labels[k], c=colors[k], marker='.')
#     axes[2].set_ylabel('Bias [$mm$]')
#     axes[2].set_ylim(-1, 0.3)
#     axes[2].plot([-1, 8], [0, 0], linestyle='dashed', c='grey')

# for ax in axes:
#     ax.set_xlabel('Lead Day')
#     ax.set_xlim(1, 7)
#     ax.legend()
#     ax.grid(alpha=0.3)

# fig.subplots_adjust(left=0.07, right=0.99, bottom=0.15, top=0.97, wspace=0.28, hspace=0)
# plt.savefig(figure_dir / 'crps_rmse_bias_by_leads.png', dpi=300)
# plt.close(fig)

## CRPS Decomposition

In [10]:
import random

from utils.crps import crps_hersbach_decomposition

In [9]:
def compute_unit(day):
    plot_data = {
        'cd_ens': [],
        'anen_ens': [],
    }

    total_days = results['obs'].shape[0]
    
    for lead_day in range(7):
        for k in plot_data.keys():
            f_tmp = results[k][:, day, lead_day]
            o_tmp = results['obs'][day, lead_day]
            
            mask = ~np.isnan(f_tmp[0])
            f_tmp = f_tmp[:, mask]
            o_tmp = o_tmp[mask]
            assert not np.isnan(f_tmp.sum()) and not np.isnan(o_tmp.sum())
            
            crps_decomp = crps_hersbach_decomposition(
                forecasts=f_tmp,
                observations=o_tmp,
                member_axis=0,
            )
    
            crps_decomp['lead_day'] = lead_day
            
            plot_data[k].append(crps_decomp)
        
    plot_data = pd.concat([pd.DataFrame(v).assign(label=k) for k, v in plot_data.items()])
    return plot_data

In [17]:
batch_plot_data = process_map(compute_unit, range(results['obs'].shape[0]), max_workers=20, chunksize=1)

  0%|          | 0/366 [00:00<?, ?it/s]

In [57]:
plot_names = ['crps', 'reliability', 'crps_potential']
verify_keys = ['anen', 'cd']

fig, axes = plt.subplots(1, len(plot_names), figsize=(10, 3))

panel_label = {
    'crps': 'CRPS [$mm$]',
    'reliability': '$Reli$ [$mm$]',
    'crps_potential': '$CRPS_{pot} = U - Resol$ [$mm$]',
}

for name, ax, axis_label in zip(plot_names, axes, ['(a)', '(b)', '(c)']):
    new = {
        k: np.array(
            [
                i[i.label==k].sort_values('lead_day')[name]
                for i in batch_plot_data
            ]
        )
        for k in ['anen_ens', 'cd_ens']
    }

    bplot(new, ax, panel_label[name])
    ax.set_xlabel("Lead Day")
    ax.text(0.01, 0.86, axis_label, ha='left', va='top', transform=ax.transAxes)

fig.subplots_adjust(left=0.06, right=0.99, bottom=0.15, top=0.99, wspace=0.25)
plt.savefig(figure_dir / 'crps_decomp.png', dpi=300)
plt.close(fig)

AnEn: [0.07169999029989516, 0.07587696739408793, 0.07616394518641383, 0.08826243136693787, 0.09265333800015876, 0.11563931713126227, 0.1273026693070134]
CorrDiff: [0.09827539421064424, 0.08541180028980272, 0.06945506736635085, 0.267706451859964, 0.09753755718191656, 0.1144815600028853, 0.12041591161083091]
AnEn: [0.032894438849646476, 0.02585933000607521, 0.0295142277965589, 0.031385808803378795, 0.03934617414143419, 0.04997040535166813, 0.06281850089952681]
CorrDiff: [0.0643827490727297, 0.05944832181854465, 0.048742965294634996, 0.23687830817408262, 0.059623133653143104, 0.06822214781850086, 0.07416606250974322]
AnEn: [0.03290992401166899, 0.03080448705590335, 0.03018574265150999, 0.032674637914307574, 0.03228921709142171, 0.03179545583333557, 0.036219508171957085]
CorrDiff: [0.020864167181281586, 0.017348248801204925, 0.016981637430358278, 0.027318667369026634, 0.016193410399030723, 0.017059175271794912, 0.022990637069821573]


## Reliability Diagram (50, 75, 95-th)

In [115]:
from utils.rel import rel_calc, rel_plot

In [117]:
fig, axes = plt.subplots(1, 3, figsize=(9.4, 3))
for thres, ax, ax_label in zip(
    np.quantile(results['obs'][results['obs'] > 0], [0.75, 0.9, 0.99]),
    axes,
    ['(a)\n75-th perc', '(b)\n90-th perc', '(c)\n99-th perc']
):
    ax.plot([0, 1], [0, 1], 'k--')
    rel_data = rel_calc(results['anen_ens'], results['obs'], threshold=thres, member_axis=0)
    ax.plot(rel_data['forecast_prob'], rel_data['observed_freq'], 'o-',
            markersize=3, label=labels['anen'], c=colors['anen'])
    rel_data = rel_calc(results['cd_ens'], results['obs'], threshold=thres, member_axis=0)
    ax.plot(rel_data['forecast_prob'], rel_data['observed_freq'], 'o-',
            markersize=3, label=labels['cd'], c=colors['cd'])
    ax.axhline(rel_data['climatology'], color='gray', linestyle=':', alpha=0.7)
    ax.axvline(rel_data['climatology'], color='gray', linestyle=':', alpha=0.7)
    ax.set_xlabel('Forecast Probability')
    ax.set_ylabel('Observed Frequency')
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper left')
    ax.set_aspect('equal')
    ax.text(0.99, 0.01, ax_label + f' ({thres:.03} mm)', ha='right', va='bottom', transform=ax.transAxes)

fig.subplots_adjust(left=0.06, right=0.99, bottom=0.13, top=0.99, wspace=0.25)
plt.savefig(figure_dir / 'rel.png', dpi=300)
plt.close(fig)

## Rank Histogram of AnEn and CD Ensemble

In [8]:
from utils import rh as rh_func

verify_keys = ['anen_ens', 'cd_ens']

# Rank histogram
rh = {
    k: rh_func(results[k], results['obs'])
    for k in verify_keys
}

In [9]:
M = results['cd_ens'].shape[0]

fig, axes = plt.subplots(1, 2, figsize=(4.9, 2.3))

axes[0].bar(range(M+1), rh['anen_ens'][0], width=0.8, edgecolor='black', facecolor=colors['anen_ens'])
axes[0].text(0.99, 0.99, f"{labels['anen_ens']}\nMRE: {rh['anen_ens'][1]:0.3f}", ha='right', va='top', transform=axes[0].transAxes)

axes[1].bar(range(M+1), rh['cd_ens'][0], width=0.8, edgecolor='black', facecolor=colors['cd_ens'])
axes[1].text(0.99, 0.99, f"{labels['cd_ens']}\nMRE: {rh['cd_ens'][1]:0.3f}", ha='right', va='top', transform=axes[1].transAxes)

for ax in axes:
    ax.set_xlabel('Ranked Member')
    ax.set_ylabel('Frequency')
    ax.set_ylim(0, 0.6)
    ax.set_xticks([i + 0.5 for i in range(0, M+1, 2)])
    ax.set_xticklabels(range(1, M+2, 2))
    ax.grid(axis='y', alpha=0.7)

fig.subplots_adjust(left=0.12, right=0.99, bottom=0.2, top=0.96, wspace=0.3, hspace=0)

axes[0].text(0.01, 0.99, '(a)', ha='left', va='top', transform=axes[0].transAxes)
axes[1].text(0.01, 0.99, '(b)', ha='left', va='top', transform=axes[1].transAxes)

plt.savefig(figure_dir / 'RH.png', dpi=300)
plt.close(fig)

## Calibration of Precipitation Intensity with Scatterplots

In [10]:
# verify_keys = ['gc', 'anen', 'unet', 'cd']

# for lead_idx in range(7):
#     fig, axes = plt.subplots(1, 4, figsize=(10, 2.6))
    
#     day_obs = np.nanmean(results['obs'], axis=(2, 3))
    
#     for plot_idx, k in enumerate(verify_keys):
#         axes[plot_idx].plot([0, 40], [0, 40], linestyle='dashed', c='grey')
        
#         day = np.nanmean(results[k], axis=(2, 3))
#         axes[plot_idx].scatter(day_obs[:, lead_idx], day[:, lead_idx], edgecolor='none', c='black', s=10)
    
#         axes[plot_idx].set_xlim(0, 40)
#         axes[plot_idx].set_ylim(0, 40)
        
#         axes[plot_idx].set_xlabel('PRISM [$mm$]')
#         axes[plot_idx].set_ylabel(f'{labels[k]} [$mm$]')
#         axes[plot_idx].text(0.99, 0.01, f"({string.ascii_lowercase[plot_idx]})", transform=axes[plot_idx].transAxes, ha='right', va='bottom')
#         axes[plot_idx].grid(linestyle='dotted')
    
#         corr_matrix = np.corrcoef(day_obs[:, lead_idx], day[:, lead_idx])
#         r = corr_matrix[0, 1]
#         axes[plot_idx].text(0.01, 0.99, f"$r$ = {r:.3f}", transform=axes[plot_idx].transAxes, ha='left', va='top')
    
#     fig.subplots_adjust(left=0.06, right=0.99, bottom=0.18, top=0.97, wspace=0.3)
#     file = figure_dir / f'Scaterplot_LeadTime_{lead_idx}.png'
#     plt.savefig(file, dpi=300)
#     plt.close(fig)

## RMSE by Precipitation Intensity

In [120]:
def plot(ax):
    x = (bins[:-1] + bins[1:]) / 2
    
    for k in verify_keys:
        errs = []
    
        for l, r in zip(bins[:-1], bins[1:]):
            obs_lead_time = results['obs'][:, lead_time_idx]
            fcst_lead_time = results[k][:, lead_time_idx]
            
            mask = (l <= obs_lead_time) & (obs_lead_time < r)
            err = np.sqrt(np.nanmean((fcst_lead_time[mask] - obs_lead_time[mask]) ** 2))
            errs.append(err)
            
        ax.plot(x, errs, label=labels[k], marker='.', c=colors[k])
        
    ax.grid(alpha=0.5)
    ax.legend()
    ax.set_xlabel('PRISM [$mm$]')
    ax.set_ylabel('RMSE [$mm$]')

for lead_time_idx in range(7):
    verify_keys = ['gc', 'anen', 'unet', 'cd']
    
    fig, axes = plt.subplots(1, 2, figsize=(7, 3))
    
    bins = np.array([0, 1, 10, 25, 50, 100, 150, 200, 300])
    plot(axes[0])
    
    # bins = np.array([0, 1, 5, 10, 15, 20])
    bins = np.array([0, 5, 10, 20, 30, 40])
    plot(axes[1])

    rect = mpl.patches.Rectangle((0, 0), 40, 25,
                                 facecolor='none',
                                 edgecolor='darkred',
                                 linestyle='--',
                                 linewidth=1)
    
    axes[0].add_patch(rect)
    
    axes[0].text(0.99, 0.01, f'Lead Day {lead_time_idx + 1}\n(a) Full Range',
                 ha='right', va='bottom', transform=axes[0].transAxes)
    axes[1].text(0.01, 0.99, '(b) Zoomed In', ha='left', va='top', transform=axes[1].transAxes)
    
    fig.subplots_adjust(left=0.09, right=0.99, bottom=0.17, top=0.97, wspace=0.25)
    plt.savefig(figure_dir / f'BinnedRMSE_LeadTime_{lead_time_idx+1}.png', dpi=300)
    plt.close(fig)

In [134]:
def plot(ax, loc=None):
    x = (bins[:-1] + bins[1:]) / 2
    
    for k in verify_keys:
        errs = []
    
        for l, r in zip(bins[:-1], bins[1:]):
            obs_lead_time = results['obs'][:, lead_time_idx]
            fcst_lead_time = results[k][:, lead_time_idx]
            
            mask = (l <= obs_lead_time) & (obs_lead_time < r)
            err = np.sqrt(np.nanmean((fcst_lead_time[mask] - obs_lead_time[mask]) ** 2))
            errs.append(err)
            
        ax.plot(x, errs, label=labels[k], marker='.', c=colors[k])
        
    ax.grid(alpha=0.5)
    if loc:
        ax.legend(loc=loc)
    else:
        ax.legend()
        

fig, axes_all = plt.subplots(3, 2, figsize=(7, 7))
mylabels = ['(a)', '(b)', '(c)', '(d)', '(e)', '(f)']
counter = 0
for lead_time_idx in [0, 2, 6]:
    verify_keys = ['gc', 'anen', 'unet', 'cd']

    if lead_time_idx == 0:
        axes = axes_all[0]
    elif lead_time_idx == 2:
        axes = axes_all[1]
    else:
        axes = axes_all[2]
    
    bins = np.array([0, 1, 10, 25, 50, 100, 150, 200, 300])
    plot(axes[0])
    
    # bins = np.array([0, 1, 5, 10, 15, 20])
    bins = np.array([0, 5, 10, 20, 30, 40])
    plot(axes[1], loc='lower right')

    rect = mpl.patches.Rectangle((0, 0), 40, 25,
                                 facecolor='none',
                                 edgecolor='darkred',
                                 linestyle='--',
                                 linewidth=1)
    
    axes[0].add_patch(rect)
    
    axes[0].text(0.99, 0.01, f'Lead Day {lead_time_idx + 1}\n{mylabels[counter]} Full Range',
                 ha='right', va='bottom', transform=axes[0].transAxes)
    counter += 1
    axes[1].text(0.01, 0.99, f'{mylabels[counter]} Zoomed In', ha='left', va='top', transform=axes[1].transAxes)
    counter += 1
    
for ax in axes_all[2, :]:
    ax.set_xlabel('PRISM [$mm$]')
for ax in axes_all[:, 0]:
    ax.set_ylabel('RMSE [$mm$]')
    
fig.subplots_adjust(left=0.09, right=0.99, bottom=0.07, top=0.99, wspace=0.2, hspace=0.15)
plt.savefig(figure_dir / f'CombinedBinnedRMSE.png', dpi=300)
plt.close(fig)

## Threat Score by Thresholds (Replaced by Brier)

In [12]:
# def threat_score(forecast, observation, threshold):
#     """
#     Calculate Threat Score (Critical Success Index) for precipitation verification.
#     """
#     m = np.isfinite(forecast) & np.isfinite(observation)
#     forecast = forecast[m]
#     observation = observation[m]
    
#     forecast = forecast > threshold
#     observation = observation > threshold

#     TP = sum((forecast == 1) & (observation == 1))
#     FP = sum((forecast == 1) & (observation == 0))
#     FN = sum((forecast == 0) & (observation == 1))
    
#     denominator = TP + FP + FN
#     if denominator == 0:
#         return float('nan')  # No events to verify
    
#     ts = TP / denominator
#     return ts

In [13]:
# values = results['obs']
# values = values[values > 0.1]

In [14]:
# for perc in [50, 75, 90, 95, 99]:
#     threshold = np.percentile(values, perc)
#     verify_keys = ['gc', 'anen', 'unet', 'cd']
    
#     fig, ax = plt.subplots(1, 1, figsize=(5, 4))
    
#     for k in verify_keys:
#         ts_by_lead_time = []
#         for lead_idx in range(len(results['lead_time'])):
#             ts = threat_score(results[k][:, lead_idx], results['obs'][:, lead_idx], threshold)
#             ts_by_lead_time.append(ts)
            
#         ax.plot(results['lead_time'], ts_by_lead_time, label=labels[k], marker='.')
        
#     ax.set_xlabel('Lead day')
#     ax.set_ylabel(f'Threat Score [> {threshold:0.1f} $mm$; {perc}-th perc]')
#     ax.grid(alpha=0.7)
#     ax.legend()
    
#     fig.subplots_adjust(left=0.13, right=0.97, bottom=0.15, top=0.97)
#     plt.savefig(figure_dir / f'TS_Perc_{perc}.png', dpi=300)
#     plt.close(fig)

## Brier Score

In [11]:
def brier_score(forecast, observation, threshold):
    if forecast.shape == observation.shape:
        forecast = (forecast > threshold).astype(np.float32)
    else:
        assert forecast.shape[1:] == observation.shape
        forecast = np.nanmean(forecast > threshold, axis=0)
    
    observation = (observation > threshold).astype(np.float32)
    brier = np.nanmean((forecast - observation) ** 2)
    return brier

In [12]:
ds_prism = xr.open_zarr('/home/myid/wh63884/data/derived/PRISM/PRISM_daily_stable_westus_20190101_20241231.zarr/')
da_ppt = ds_prism.ppt.where(ds_prism.ppt > 0.0, np.nan).to_numpy()

In [13]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

letters = ['a', 'b', 'c']

for plot_idx, perc in enumerate([50, 75, 95]):
    threshold = np.nanquantile(da_ppt, perc/100, axis=0)
    verify_keys = ['gc', 'anen_ens', 'unet', 'cd_ens']
    
    for k in verify_keys:
        brier_by_lead_time = []
        for lead_idx in range(len(results['lead_time'])):
            f = results[k][:, lead_idx] if k in ['gc', 'unet'] else results[k][:, :, lead_idx]
            ts = brier_score(f, results['obs'][:, lead_idx], threshold[None] if k in ['gc', 'unet'] else threshold[None, None])
            brier_by_lead_time.append(ts)
            
        axes[plot_idx].plot(results['lead_time'], brier_by_lead_time, label=labels[k], marker='.', c=colors[k])
        
    axes[plot_idx].set_xlabel('Lead day')
    axes[plot_idx].set_ylabel(f'Brier Score')
    axes[plot_idx].text(0.99, 0.01, f'({letters[plot_idx]}) > {perc}-th perc',
                        ha='right', va='bottom', transform=axes[plot_idx].transAxes)
    axes[plot_idx].grid(alpha=0.5)
    axes[plot_idx].legend()
    
fig.subplots_adjust(left=0.06, right=0.99, bottom=0.15, top=0.97, wspace=0.3)
plt.savefig(figure_dir / f'Brier.png', dpi=300)
plt.close(fig)

/home/myid/wh63884/miniconda3/envs/venv_torch/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1634: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,


## Spread-Skill Correlation

In [14]:
verify_keys = ['anen_ens', 'cd_ens']

ss = {
    k: {
        'std': np.nanmean(np.nanstd(results[k], axis=0), axis=(2, 3)),
        'rmse': np.sqrt(np.nanmean((np.nanmean(results[k], axis=0) - results['obs']) ** 2, axis=(2, 3))),
    } for k in verify_keys
}

/home/myid/wh63884/miniconda3/envs/venv_torch/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipykernel_1062416/2191712831.py:6: RuntimeWarning: Mean of empty slice
  'rmse': np.sqrt(np.nanmean((np.nanmean(results[k], axis=0) - results['obs']) ** 2, axis=(2, 3))),


In [15]:
fig, axes = plt.subplots(1, 3, figsize=(9, 2.8))

for ax in axes:
    ax.axline((0, 0), (1, 1), linestyle='dashed', c='darkred')

for plot_idx, lead_idx in enumerate([0, 2, 6]):
    for k, v in ss.items():
    
        std = v['std'][:, lead_idx]
        rmse = v['rmse'][:, lead_idx]
        
        bins = np.percentile(rmse, range(0, 101, 10))
        
        x, y = [], []
        
        for l, r in zip(bins[:-1], bins[1:]):
            m = (l <= rmse) & (rmse < r)
            
            x.append(np.mean(std[m]))
            y.append(np.mean(rmse[m]))
            
        axes[plot_idx].plot(x, y, label=labels[k], c=colors[k], marker='.')
        msg = f'${labels[k]}: {np.corrcoef(x, y)[0, 1]:0.3f}$'
        
axes[0].text(0.01, 0.99, '(a) Lead Day 1', ha='left', va='top', transform=axes[0].transAxes)
axes[1].text(0.01, 0.99, '(b) Lead Day 3', ha='left', va='top', transform=axes[1].transAxes)
axes[2].text(0.01, 0.99, '(c) Lead Day 7', ha='left', va='top', transform=axes[2].transAxes)

for ax in axes:
    ax.set_xlabel('Spread [$\sigma$, $mm$]')
    ax.set_ylabel('RMSE [$mm$]')
    ax.legend(loc='lower right')
    ax.grid(alpha=0.7)

fig.subplots_adjust(left=0.06, right=0.99, bottom=0.17, top=0.97, wspace=0.25)
plt.savefig(figure_dir / f'SS.png', dpi=300)
plt.close(fig)

## Power Spectral Density

In [4]:
import numpy as np

def compute_psd_2d(field, dx=4.0, dy=4.0):
    """
    Compute isotropic 1D PSD from a 2D field with correct normalization.
    dx, dy: grid spacing in physical units (e.g., meters, km).
    """
    # Remove NaNs and mean
    field = np.nan_to_num(field, nan=0.0)
    field = field - np.mean(field)

    ny, nx = field.shape

    # 2D FFT
    F = np.fft.fft2(field)
    F = np.fft.fftshift(F)

    # Normalized 2D PSD
    psd2D = (np.abs(F)**2) * (dx * dy) / (nx * ny)**2

    # Wavenumber coordinates
    kx = np.fft.fftshift(np.fft.fftfreq(nx, dx))  # cycles per unit distance
    ky = np.fft.fftshift(np.fft.fftfreq(ny, dy))
    KX, KY = np.meshgrid(kx, ky)
    K = np.sqrt(KX**2 + KY**2)  # isotropic wavenumber magnitude

    # Radial bins in k-space
    k_bins = np.linspace(0, K.max(), min(nx, ny)//2 + 1)
    psd_1d = np.zeros(len(k_bins)-1)
    k_bin_centers = 0.5 * (k_bins[:-1] + k_bins[1:])

    # Bin averaging with proper area normalization
    for i in range(len(k_bins)-1):
        mask = (K >= k_bins[i]) & (K < k_bins[i+1])
        if np.any(mask):
            # Sum power in ring and divide by ring width in k-space
            bin_area = np.pi * (k_bins[i+1]**2 - k_bins[i]**2)
            psd_1d[i] = psd2D[mask].sum() / bin_area
        else:
            psd_1d[i] = np.nan

    return k_bin_centers, psd_1d

def compute_psd_batch(data, dx=4.0, dy=4.0):
    """
    data: shape [n_days, height, width]
    """
    n_days = data.shape[0]
    psd_list = []
    for i in range(n_days):
        k, psd = compute_psd_2d(data[i], dx, dy)
        psd_list.append(psd)
    psd_array = np.array(psd_list)
    mean_psd = np.nanmean(psd_array, axis=0)
    return k, mean_psd

In [5]:
lead_idx = 0
verify_keys = ['obs', 'gc', 'anen', 'unet', 'cd']

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))

for k in verify_keys:
    # idx = np.nansum(results[k][:, lead_idx], axis=(1, 2)).argsort()[-5:]
    # d = results[k][idx, lead_idx]
    
    d = results[k][:, lead_idx]
    k_bin_centers, psd = compute_psd_batch(d, dx=4.0, dy=4.0)

    for ax in axes:
        ax.loglog(k_bin_centers, psd, label=labels[k], c=colors[k])
    
for ax in axes:
    ax.legend(loc='lower left')
    ax.set_xlabel('Spatial Frequency [$km^{-1}$]')
    ax.set_ylabel('Power Spectral Density [$mm^2/km^2$]')
    ax.grid(alpha=0.7)

axes[1].set_xlim(0.04, 0.3)
axes[1].set_ylim(0.15, 3e3)

rect = mpl.patches.Rectangle((0.04, 0.1), 0.3, 3000,
                             facecolor='none',
                             edgecolor='darkred',
                             linestyle='--',
                             linewidth=1)

axes[0].add_patch(rect)

axes[0].text(0.99, 0.99, '(a) Full Range', ha='right', va='top', transform=axes[0].transAxes)
axes[1].text(0.99, 0.99, '(b) Zoomed In', ha='right', va='top', transform=axes[1].transAxes)

fig.subplots_adjust(left=0.09, right=0.96, bottom=0.13, top=0.99)
plt.savefig(figure_dir / f'Power.png', dpi=300)
plt.close(fig)

## Study Area

In [1]:
import cmocean
import rioxarray

import xarray as xr
import matplotlib as mpl
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import cartopy.feature as cfeature

from utils.presets import Precipitation

cmap, norm = Precipitation.cmap_norm()

In [2]:
file = "~/data/derived/PRISM/PRISM_daily_stable_westus_20190101_20241231.zarr/"
ds = xr.open_zarr(file)

ds_ppt_mean = ds.ppt.mean('date')

# Load DEM from local GeoTIFF
dem = rioxarray.open_rasterio("/home/myid/wh63884/data/original/Oro/exportImage.tiff").squeeze()

# Extract ppt coords and bounds
lons = ds_ppt_mean['x'].values
lats = ds_ppt_mean['y'].values
lon_min, lon_max = float(lons.min()), float(lons.max())
lat_min, lat_max = float(lats.min()), float(lats.max())

In [12]:
fig, axes = plt.subplots(
    1, 2, figsize=(7.8, 4.6),
    subplot_kw={'projection': ccrs.PlateCarree()}
)

# Left panel: Elevation
e0 = axes[0].pcolormesh(
    dem['x'], dem['y'], dem,
    cmap='terrain',
    transform=ccrs.PlateCarree()
)
axes[0].set_extent([lon_min, lon_max, lat_min, lat_max])
axes[0].coastlines()
axes[0].add_feature(cfeature.BORDERS, linewidth=0.5)
axes[0].add_feature(cfeature.STATES, linewidth=0.5)
cbar = fig.colorbar(e0, ax=axes[0], orientation='horizontal', pad=0.05)
cbar.set_label("Topography & Bathymetry [$m$]")

# Right panel: Precipitation
p1 = axes[1].pcolormesh(
    ds_ppt_mean['x'], ds_ppt_mean['y'], ds_ppt_mean,
    transform=ccrs.PlateCarree(),
    # cmap=cmocean.cm.rain,
    norm=mpl.colors.Normalize(vmin=0, vmax=10), cmap=cmap,
)
axes[1].set_extent([lon_min, lon_max, lat_min, lat_max])
axes[1].coastlines()
axes[1].add_feature(cfeature.BORDERS, linewidth=0.5)
axes[1].add_feature(cfeature.STATES, linewidth=0.5)
cbar = fig.colorbar(p1, ax=axes[1], orientation='horizontal', pad=0.05)
cbar.set_label("Mean Areal Precipitation [$mm/day$]")

from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import numpy as np

for ax in axes:
    # Define tick locations
    lon_ticks = np.arange(np.ceil(lon_min), np.floor(lon_max)+1, 2)  # every 2°
    lat_ticks = np.arange(np.ceil(lat_min), np.floor(lat_max)+1, 2)

    ax.set_xticks(lon_ticks, crs=ccrs.PlateCarree())
    ax.set_yticks(lat_ticks, crs=ccrs.PlateCarree())

    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())

    ax.tick_params(labelsize=8)  # optional: adjust font size

fig.subplots_adjust(0.05, 0.01, 0.98, 0.99, wspace=0.12)
# plt.show()
plt.savefig("Figures_ED/PRISM_Study_Area.png", dpi=350)
plt.close(fig)

## AR Days Evaluation

In [125]:
valid_times = [
    pd.to_datetime('2024-8-24').date(),
    pd.to_datetime('2024-2-1').date(),
    pd.to_datetime('2024-1-13').date(),
    pd.to_datetime('2024-3-10').date(),
    pd.to_datetime('2024-1-27').date(),
    pd.to_datetime('2024-3-2').date(),
    pd.to_datetime('2024-1-31').date(),
    pd.to_datetime('2024-6-3').date(),
    pd.to_datetime('2024-3-27').date(),
    pd.to_datetime('2024-1-24').date(),
    pd.to_datetime('2024-3-12').date(),
    pd.to_datetime('2024-5-4').date(),
    pd.to_datetime('2024-1-17').date(),
    pd.to_datetime('2024-2-17').date(),
    pd.to_datetime('2023-12-29').date(),
    pd.to_datetime('2024-3-6').date(),
    pd.to_datetime('2023-12-22').date(),
    pd.to_datetime('2024-3-22').date(),
    pd.to_datetime('2024-1-22').date(),
    pd.to_datetime('2023-12-18').date(),
    pd.to_datetime('2024-1-20').date(),
    pd.to_datetime('2024-2-19').date(),
    pd.to_datetime('2024-3-30').date(),
    pd.to_datetime('2023-11-16').date(),
    pd.to_datetime('2024-2-4').date(),
    pd.to_datetime('2024-2-15').date(),
    pd.to_datetime('2024-1-2').date(),
]

In [126]:
len(valid_times)

27

In [127]:
day_indices = np.array([np.where(results['forecast_init_time'] + pd.to_timedelta(1, unit='d') == v)[0].item() for v in valid_times])

In [131]:
partial_results = {}
verify_keys = ['gc', 'anen', 'unet', 'cd']

for k in verify_keys + ['obs']:
    if k not in partial_results:
        partial_results[k] = []

    for l in range(1):
        partial_results[k].append(results[k][day_indices-l, l])
        
    partial_results[k] = np.stack(partial_results[k], axis=1)

# partial_results['obs'][partial_results['obs'] < 20] = np.nan

In [132]:
rows = []
for thres in [0, 10, 50, 100]:
    mask = partial_results['obs'] >= thres
    row = {k: (np.nanmean((partial_results[k][mask] - partial_results['obs'][mask]) ** 2)) ** 0.5 for k in verify_keys}
    row['thres'] = thres
    rows.append(row)
rows = pd.DataFrame(rows)
rows = rows[['thres', 'gc', 'anen', 'unet', 'cd']]
rows.columns = ['Threshold [$mm$]', labels['gc'], labels['anen'], labels['unet'], labels['cd']]

In [133]:
rows

,Threshold [$mm$],GraphCast,AnEn,Unet,CorrDiff
0,0,4.767980,4.498131,5.123917,4.709330
1,10,11.902314,11.141646,13.059023,11.750624
2,50,29.760702,25.638490,29.772207,25.125759
3,100,77.123337,66.137666,68.953110,62.811897


## AnEn Weights Experiments

In [9]:
from glob import glob

verify_results = {}

for file_anen_exp in tqdm(sorted(glob("./tmp/anen/paper_review/*.nc"))):
    with xr.open_dataset(file_anen_exp) as ds_anen_exp:
        exp_tag = file_anen_exp.split('/')[-1].rstrip('.nc').split('_')[-1]
        print(f'Extracted the experiment tag: {exp_tag}')
        verify_results[exp_tag] = np.nanmean(ds_anen_exp['analogs'].to_numpy()[:15], axis=0)
        verify_results[exp_tag] = verify_results[exp_tag].reshape(*verify_results[exp_tag].shape[:2], 228, 216).transpose(1, 0, 2, 3)

  0%|          | 0/5 [00:00<?, ?it/s]

Extracted the experiment tag: exp1


/tmp/ipykernel_308746/595754565.py:9: RuntimeWarning: Mean of empty slice
  verify_results[exp_tag] = np.nanmean(ds_anen_exp['analogs'].to_numpy()[:15], axis=0)


Extracted the experiment tag: exp2


/tmp/ipykernel_308746/595754565.py:9: RuntimeWarning: Mean of empty slice
  verify_results[exp_tag] = np.nanmean(ds_anen_exp['analogs'].to_numpy()[:15], axis=0)


...

/tmp/ipykernel_308746/595754565.py:9: RuntimeWarning: Mean of empty slice
  verify_results[exp_tag] = np.nanmean(ds_anen_exp['analogs'].to_numpy()[:15], axis=0)


Extracted the experiment tag: exp4


/tmp/ipykernel_308746/595754565.py:9: RuntimeWarning: Mean of empty slice
  verify_results[exp_tag] = np.nanmean(ds_anen_exp['analogs'].to_numpy()[:15], axis=0)


Extracted the experiment tag: exp5


/tmp/ipykernel_308746/595754565.py:9: RuntimeWarning: Mean of empty slice
  verify_results[exp_tag] = np.nanmean(ds_anen_exp['analogs'].to_numpy()[:15], axis=0)


In [10]:
verify_results['anen'] = results['anen']
verify_results['cd'] = results['cd']

In [11]:
for k, v in verify_results.items():
    print(f"{k}: {v.shape}")

exp1: (366, 7, 228, 216)
exp2: (366, 7, 228, 216)
exp3: (366, 7, 228, 216)
exp4: (366, 7, 228, 216)
exp5: (366, 7, 228, 216)
anen: (366, 7, 228, 216)
cd: (366, 7, 228, 216)


In [36]:
## Table for RMSE calculated for each method and each lead time
rmses = {k: np.nanmean((v - results['obs']) ** 2, axis=(2, 3)) ** 0.5 for k, v in verify_results.items()}

In [157]:
verify_keys = rmses.keys()
fig, ax = plt.subplots(1, 1, figsize=(6.7, 3))
bplot(rmses, ax, 'RMSE [$mm$]', rmses.keys(), 0.12)
ax.set_xlim(-0.5, 6.5)
ax.set_xlabel("Lead Day")
fig.subplots_adjust(left=0.06, right=0.99, bottom=0.15, top=0.99)
plt.savefig(figure_dir / 'exps_rmse_by_leads_boxplot_anen.png', dpi=300)
plt.close(fig)

AnEn Exp. 1: [0.4310025862476694, 0.47579697995772685, 0.49689636538353577, 0.5575403699620678, 0.669074374381812, 0.8157453708967446, 0.9148264397273173]
AnEn Exp. 2: [0.4243098300829652, 0.4869262688008862, 0.49078433443956826, 0.5811450638810207, 0.6777295888613273, 0.7750616972370612, 0.8631354926711803]
AnEn Exp. 3: [0.44232824963675693, 0.46599786719360037, 0.5055410057185252, 0.5682496875339191, 0.6612024796254572, 0.7775323579188925, 0.8883418002464181]
AnEn Exp. 4: [0.4375588988030135, 0.4826735970898965, 0.4940696772247485, 0.5588490415669117, 0.6741888494159505, 0.8081240556176414, 0.8908385920669377]
AnEn Exp. 5: [0.429007158378301, 0.46817551862190826, 0.4936177171553573, 0.5560517369335829, 0.6702857472623533, 0.824529891760692, 0.8982524716299458]
AnEn: [0.42952499761892615, 0.46774774126248253, 0.4817936359231181, 0.5488760670780228, 0.6237917952841321, 0.7734931654573507, 0.8978342917844944]
CorrDiff: [0.42159366607666016, 0.468960702419281, 0.4333403706550598, 0.57160

In [155]:
def plot(ax, loc=None):
    x = (bins[:-1] + bins[1:]) / 2
    
    for k in verify_keys:
        errs = []
    
        for l, r in zip(bins[:-1], bins[1:]):
            obs_lead_time = results['obs'][:, lead_time_idx]
            fcst_lead_time = verify_results[k][:, lead_time_idx]
            
            mask = (l <= obs_lead_time) & (obs_lead_time < r)
            err = np.sqrt(np.nanmean((fcst_lead_time[mask] - obs_lead_time[mask]) ** 2))
            errs.append(err)

        if k in labels:
            label_key = labels[k]
            color_key = colors[k]
        elif k.startswith('exp'):
            label_key = labels['anen'] + ' ' + k.replace('exp', 'Exp. ')
            color_key = shade(colors['anen'], 0.2+0.1 * float(label_key[-1]))
        else:
            raise Exception(f'Unknown key: {k}')
            
        ax.plot(x, errs, label=label_key, marker='.', c=color_key)
        
    ax.grid(alpha=0.5)
    if loc:
        ax.legend(loc=loc)
    else:
        ax.legend()
        

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
mylabels = ['(a)', '(b)', '(c)']
counter = 0

for lead_time_idx in [0, 2, 6]:
    verify_keys = verify_results.keys()

    if lead_time_idx == 0:
        ax = axes[0]
    elif lead_time_idx == 2:
        ax = axes[1]
    else:
        ax = axes[2]
    
    bins = np.array([0, 1, 10, 25, 50, 100, 150, 200, 300])
    plot(ax)
    
    ax.text(0.99, 0.01, f'{mylabels[counter]} Lead Day {lead_time_idx + 1}',
            ha='right', va='bottom', transform=ax.transAxes)
    counter += 1

axes[0].set_ylabel('RMSE [$mm$]')
for ax in axes:
    ax.set_xlabel('PRISM [$mm$]')
    
fig.subplots_adjust(left=0.06, right=0.99, bottom=0.145, top=0.99, wspace=0.17, hspace=0.15)
plt.savefig(figure_dir / f'exps_CombinedBinnedRMSE.png', dpi=300)
plt.close(fig)